In [ ]:
#Importación de librerías

import os
import json
import random
from openai import AsyncOpenAI
import asyncio
import difflib
import unicodedata
import re
import pandas as pd
import yaml
import re

## Construcción de la batería de preguntas.

Hacer chunking justo antes de hacer las preguntas, puede introducir cierto sesgo, ya que estaría haciendo las preguntas sobre partes concretas del documento, en lugar de sacarlas de partes cualesquiera. Esto si luego monto el sistema RAG con el mismo chunking, no supondría un problema, de hecho podría mejorar la precisión del sistema. Pero sin embargo, si luego a posteriori quisiera hacer cierta experimentación variando el chunking para comparar el desempeño del RAG según ese criterio, estoy seguro que el sistema RAG construido con el mismo chunking a partir del cual se hicieron las preguntas, va a tener una mayor precisión que los otros sistemas RAG con un chunking distinto.

RAG, chunk-level evaluation bias; Al no hacer chunking previamente evitamos tener una configuración del RAG que esté estrechamente alineada con el golden Dataset de preguntas. Así cualquier estrategia de chunking del RAG que hagamos a posteriori, puede contener ese texto parcialmente o completamente.

## ¿Cómo determinar el número de preguntas por documento?

El enfoque más usado en la literatura: ratio fijo por palabras.

Debo definir un ratio de preguntas por palabra de documento. Dado que trabajamos con documentos de distintas extensiones, y además con páginas que pueden tener mucho contenido estructurado en tablas, lo más profesional y simple es una función basada en conteo de palabras con límites mínimo y máximo.

La referencia más directa y citable para tu TFG es el paper "Can we Evaluate RAGs with Synthetic Data?" (Thakur et al., 2024), que fija explícitamente 5 preguntas por documento como valor por defecto para documentos de ~3.500–4.300 palabras, reduciéndolo a 2 preguntas para documentos cortos (menos de ~500 palabras).

Esto implica un ratio de aproximadamente 1 pregunta por cada 800 palabras, que es el criterio más justificable científicamente para tu memoria.

Finalmente el criterio para determinar el número total de preguntas que extraemos de cada documento, será extraer 1 pregunta por cada 800 palabras, con un número mínimo de 300 palabras por documento. Para aquellos documentos de menos de 300 palabras no se extraerán preguntas.

Luego para escoger cuantas preguntas serán de cada tipo, definimos sobre el número de preguntas total determinado los siguientes porcentajes:

-34% para preguntas factual_true: Las preguntas factuales verdaderas deben ser el núcleo del dataset, nos servirán para medir el uso normal del sistema. Son preguntas que pueden responderse atendiendo expresamente al contenido de los documentos.

-33% para preguntas factual_reasoning: Son preguntas que requieren inferencia, síntesis o razonamiento sobre el documento. Permiten diferenciar claramente un RAG robusto de uno que no lo es. Las preguntas de razonamiento son las más difíciles de generar automáticamente con calidad. El LLM puede tender a crear preguntas que parecen de razonamiento pero en realidad tienen respuesta directa. Será necesario prestar especial atención a estas preguntas durante la curación manual.

Estos son los que mejor se adaptan a documentos empresariales como los de AMC Global y son evaluables con RAGAS:

Inferencia multi-fragmento: la respuesta requiere combinar información de dos o más partes del documento.

"Dado el procedimiento de solicitud de equipos y los plazos de aprobación, ¿qué ocurre si un trabajador solicita un equipo con menos tiempo del requerido?"

Comparación: el modelo debe contrastar dos conceptos, procedimientos o entidades del documento.

"¿En qué se diferencia el procedimiento de solicitud de vacaciones del de bajas médicas según el manual?"

Causal / "por qué": requiere inferir causas o consecuencias a partir de las políticas descritas.

"¿Por qué es necesaria la aprobación del gerente directo en las solicitudes de equipos?"

Condicional / "qué pasa si": fuerza al modelo a razonar sobre casos específicos descritos en el documento.

"Si un trabajador lleva menos de seis meses en la empresa, ¿cómo le afecta la política de vacaciones?"


-33% para preguntas factual_trap: Este porcentaje es suficiente para medir robustez sin sobredimensionar casos adversariales. En benchmarks populares como TruthfulQA y RobustRAG el ratio de preguntas de este tipo ronda entre 20 y 30%. El término adversarial en evaluación de LLMs/RAG hace referencia a preguntas diseñadas intencionalmente para inducir al modelo a cometer un error concreto. No significa preguntas maliciosas, sino preguntas que ejercen una presión específica sobre el sistema.las factual_trap son adversariales en el sentido de que contienen un supuesto falso implícito que el modelo debe detectar y corregir en lugar de aceptar.

Premisa falsa: "¿Cuántos días de preaviso previo son necesarios para solicitar vacaciones según el manual?" — cuando el manual dice que no hay plazo de preaviso definido, sino un procedimiento de aprobación.

Confusión numérica: "¿Es cierto que los trabajadores tienen derecho a 30 días de vacaciones al año?" — cuando el documento establece 22 días laborables.

Atribución errónea: "¿Es el departamento de RRHH quien aprueba directamente todas las solicitudes de equipos?" — cuando el documento dice que es el gerente directo.


-15% para preguntas out_of_domain: Este porcentaje de preguntas queda fuera del 100%, pero son preguntas que no requieren que sus respuestas sean explícitamente extraídas del texto del documento. Al quedar su respuesta fuera del dominio de los documentos que conforman el corpus, no queremos que sean mas de una minoría representativa, ya que demasiadas preguntas out_of_domain podrían sesgar las métricas globales de evaluación del RAG.

Es importante mencionar que la respuesta correcta para un sistema RAG de dominio cerrado (como en nuestro caso, sobre documentos internos de AMC Global) es siempre la abstención informativa: "No dispongo de información suficiente en la documentación disponible para responder esta pregunta."

El paper "Don't Hallucinate, Abstain" (Feng et al., ACL 2024) argumenta que un LLM robusto debe abstenerse cuando el conocimiento no está en su contexto recuperado, en lugar de recurrir a su conocimiento paramétrico previo.
El paper "Sufficient Context" (Loren et al., 2025) corrobora que los LLMs tienen grandes dificultades para abstenerse cuando el contexto recuperado es insuficiente, y que es precisamente uno de los fallos de alucinación más comunes en RAG.

Si el RAG responde algo concreto en lugar de abstenerse → es una alucinación paramétrica (usa conocimiento interno del LLM no grounded en el contexto). Eso es exactamente lo que quieres detectar y medir.

**Como para OOD el contexto gold está vacío o es irrelevante, simplemente deberías excluir context_recall del cálculo para preguntas OOD en tu análisis por segmentos. Esto es fácil filtrando por category_domain en tu código de evaluación.


--------------------------------------------------------------
Cabe destacar que si el documento es muy corto y según el ratio de preguntas establecido, se nos indica que el número máximo de preguntas a extraer de ese documento es menor que 3, asignamos el número de preguntas de cada tipo de forma cíclica priorizando a las factual_true y factual_reasoning.

| Valor             | Descripción                                                     |
| ----------------- | --------------------------------------------------------------- |
| factual_true      | Factual directa, respuesta extraíble literalmente               |
| factual_trap      | Factual con premisa falsa que el modelo debe corregir           |
| factual_reasoning | Requiere inferencia, síntesis o razonamiento sobre el documento |
| out_of_domain     | La respuesta no está en el corpus                               |

In [ ]:
def calcular_num_preguntas(texto_completo: str) -> dict:
    num_palabras = len(texto_completo.split())


    # Ninguna pregunta si el texto es demasiado corto para ser informativo
    if num_palabras < 300:
        return {"factual_true": 0,"factual_reasoning": 0, "factual_trap": 0, "out_of_domain": 0, "total_sin_preguntas_ood": 0, "total_con_preguntas_ood": 0
}

    # Ratio base: 1 pregunta por 800 palabras, mínimo 1, máximo 25
    total = max(1, min(25, round(num_palabras / 800)))

    if total > 2:

      n_factual_true      = max(1, round(total * 0.34))
      n_factual_trap      = max(1, round(total * 0.33))
      n_factual_reasoning = total - n_factual_true - n_factual_trap  # resto exacto
      n_factual_reasoning = max(1, n_factual_reasoning)  # garantiza al menos 1
      n_out_of_domain     = max(1, round(total * 0.15))
      #En función del total sacamos el out_of_domain;
      #Como las respuestas de las preguntas de out_of_domain, realmente no tienen que
      # extraerse del documento como tal podemos obtenerlas como un porcentaje fuera del total.
      # Así del número total de preguntas que se deben extraer del documento, sacamos las preguntas factuales,
      # pero para las de fuera del dominio que no requieren respuestas extraibles del documento, determinamos un número de preguntas a parte



    else:

      tipos = ["factual_true", "factual_reasoning", "factual_trap"]
      conteos = {"factual_true": 0, "factual_reasoning": 0, "factual_trap": 0}

      for i in range(total):
          conteos[tipos[i % 3]] += 1  # cicla entre los tres tipos

      n_factual_true      = conteos["factual_true"]
      n_factual_reasoning = conteos["factual_reasoning"]
      n_factual_trap      = conteos["factual_trap"]
      n_out_of_domain     = 0


    return {

          "factual_true": n_factual_true,
          "factual_reasoning": n_factual_reasoning,
          "factual_trap": n_factual_trap,
          "out_of_domain": n_out_of_domain,
          "total_sin_preguntas_ood": total,
          "total_con_preguntas_ood": total + n_out_of_domain,
      }



In [ ]:
    input_path = "/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/Procesamiento-Corpus/jsons_output"
    file_prefix = None

    json_files = [
        f for f in os.listdir(input_path)
        if f.endswith(".json")
        and (file_prefix is None or f.startswith(f"{file_prefix}-"))
    ]


    for json_file in sorted(json_files):
        file_path = os.path.join(input_path, json_file)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(data["nombre_original"], "pag: ",data["num_paginas"])
        diccionario_preguntas_tipo = calcular_num_preguntas(data["texto_completo"])
        print(diccionario_preguntas_tipo)

**Determinación del número de preguntas por documento**

Con el fin de garantizar que la batería de preguntas sea proporcional al contenido informativo de cada documento, se diseñó una función de asignación automática que determina el número y tipo de preguntas a generar para cada documento del corpus de AMC Global.

El criterio base adoptado es un ratio de una pregunta por cada 800 palabras, valor establecido a partir de la práctica descrita en Thakur et al. (2024), quienes emplean aproximadamente cinco preguntas para documentos de entre 3.500 y 4.300 palabras. El número total de preguntas resultante se acota con un mínimo de 1 y un máximo de 25, evitando así tanto la subrepresentación de documentos muy densos como la generación excesiva de preguntas sobre documentos breves. Los documentos con menos de 300 palabras se descartan automáticamente por no disponer de contenido suficiente para formular preguntas de calidad.

(El ratio de 1 pregunta por cada 800 palabras está diseñado para estimar cuántas preguntas con respuesta en el documento puede sostener ese texto sin repetirse ni volverse trivial. Ese ratio tiene sentido para factual_true, factual_reasoning y factual_trap porque todas dependen del contenido del documento: a más contenido, más preguntas de calidad se pueden extraer.

Las preguntas OOD no dependen del contenido del documento para ser generadas, sino de lo que el documento no cubre. Un documento de 500 palabras y uno de 10.000 palabras pueden generar el mismo número de preguntas OOD plausibles, porque lo que genera esas preguntas es el dominio temático del documento, no su extensión.)

Las preguntas se clasifican en cuatro tipos: factuales verificables (factual_true), de razonamiento (factual_reasoning), factuales con trampa (factual_trap) y fuera del dominio (out_of_domain). Los tres primeros tipos comparten la característica de requerir una respuesta extraíble directa o indirectamente del documento, por lo que su número se calcula a partir del total proporcional al documento. Las preguntas fuera del dominio, cuyas respuestas no se encuentran en el corpus, se tratan como un conjunto adicional independiente, calculado como un porcentaje complementario sobre ese mismo total, de forma que no compitan por cuota con las preguntas de contenido factual.

Para documentos con un volumen suficiente (total igual o superior a tres preguntas), la distribución entre los tipos factuales sigue las proporciones 50%, 25% y 25% respectivamente, con la restricción de que cada tipo reciba al menos una pregunta. El número de preguntas fuera del dominio se fija en un 15% adicional sobre el total factual. Para documentos cortos (total de una o dos preguntas), las preguntas fuera del dominio se suprimen por considerar que un documento de extensión reducida no ofrece suficiente contexto para generar preguntas OOD plausibles, y la asignación entre los tipos factuales se realiza mediante una distribución cíclica determinista, garantizando máxima diversidad tipológica con independencia de la aleatoriedad.



## Parte de Prompt

**¿Cómo mitigar el Sesgo de variabilidad y repetición en la generación?**

Este es un problema real, conocido en la literatura como degeneration o mode collapse en la generación de texto: el modelo tiende a repetir patrones lingüísticos y estructurales cuando genera muchos ejemplos del mismo tipo en un mismo prompt.

Hay varias formas de controlarlo, y pueden combinarse:

Desde el prompt (parcialmente): puedes incluir instrucciones del tipo "genera preguntas que cubran distintos apartados del documento, evitando repetir el mismo patrón de formulación". Esto ayuda, pero no es suficiente por sí solo para volúmenes grandes.

Desde los parámetros de generación (más efectivo): ajustar temperatura y top-p hacia valores que favorezcan diversidad, como verás en el punto de parámetros más abajo.

Usando un enfoque de 4 prompts separados (uno por cada tipo de pregunta): como explico en el siguiente punto, llamadas independientes y cortas son naturalmente menos propensas a este problema que una sola llamada larga.

Variabilidad con la semilla de la API: Semilla aleatoria (seed): la API de OpenAI permite fijar un parámetro seed para reproducibilidad, pero si quieres variedad entre documentos distintos, deberías no fijarlo o variarlo por documento, de modo que cada llamada explore un espacio de generación diferente.

**Enfoque de usar 4 prompts separados, uno pro tipo de pregunta.**

Esta idea está justificada por dos razones:

Reduce confusiones del modelo: cuando en un mismo prompt pides cuatro comportamientos distintos (factual directa, trampa, razonamiento, OOD), el modelo puede "mezclar" criterios, especialmente en la línea entre factual_trap y factual_reasoning. Prompts separados dan contexto más enfocado.

Reduce el sesgo de posición: en prompts largos con múltiples tipos, los LLMs tienen tendencia a poner más esfuerzo en los primeros ítems de cada tipo. Con prompts separados, cada tipo recibe el mismo "foco" de atención.

**Al tener 4 prompts por documento serán 4 llamadas a la API, por lo que conviene paralelizar, con asycio. El único límite que podría haber sería el ratio de peticiones por minuto que para GPT-4.1 en nivels de pago es suficientemente alto.

**Justificación de pasar el documento completo: ventana de contexto de GPT-4.1**

GPT-4.1 (versión 2025-04-14 vía API) tiene una ventana de contexto de 1.047.576 tokens, es decir, aproximadamente 1 millón de tokens.


"GPT-4.1, el modelo seleccionado para la generación del dataset, dispone de una ventana de contexto de 1.047.576 tokens (OpenAI, 2025), lo que equivale a aproximadamente 700.000 palabras. Dado que el documento más extenso del corpus de AMC Global no supera las 40.000 palabras (~65.000 tokens), es técnicamente viable procesar cada documento completo en una única llamada a la API, sin necesidad de segmentación previa."


**¿Debería evitar preguntas repetidas sobre el mismo contexto?**
Esta distinción es importante. Hay dos tipos de "repetición":

Misma pregunta, mismo contexto: totalmente a evitar. Una pregunta reformulada de otra ya existente no añade valor al dataset ni diversidad a la evaluación.

Misma sección del documento, distintas preguntas: es completamente válido. Una misma sección puede dar lugar a una factual_true y una factual_trap sobre aspectos distintos. El context_gold puede ser el mismo fragmento y eso no supone ningún problema para RAGAS, porque lo que evalúa es la respuesta del sistema, no la unicidad del contexto.

También se debe evitar es que para el mismo documento todas las preguntas se concentren en un único apartado, dejando secciones sin cubrir. Esto puedes controlarlo parcialmente desde el prompt indicando al modelo que distribuya las preguntas por distintas secciones del documento.



**Temperatura, Top-p y Top-k**


*¿Qué es cada parámetro?*


Temperatura controla la "aleatoriedad" de la distribución de probabilidad sobre el vocabulario al seleccionar cada token. Matemáticamente, divide los logits por el valor de temperatura antes de aplicar softmax:

Temperatura baja (→ 0): la distribución se vuelve muy "puntiaguda", casi siempre elige el token más probable → respuestas deterministas y repetitivas.

Temperatura alta (→ 2): la distribución se aplana → mayor diversidad, pero también mayor riesgo de incoherencia.

Top-k definie los k tokens siguientes mas probables entre los que construir una distribución de probabilidad para escoger al siguiente token de la secuencia, para así evitar determinismo en las respuestas. Restringe la selección del siguiente token a los k tokens más probables, descartando el resto antes de muestrear. Por ejemplo con top_k=50, solo los 50 tokens más probables entran en el sorteo:

top_k bajo → respuestas más conservadoras y predecibles.

top_k alto → mayor variedad léxica.

Top-p (nucleus sampling) en lugar de fijar un número fijo de candidatos, acumula tokens ordenados por probabilidad hasta alcanzar una probabilidad acumulada P. Por ejemplo con top_p=0.9:

Si los 10 tokens más probables suman ya el 90% de la probabilidad → solo esos 10 se consideran.

Si hay más diversidad y se necesitan 200 tokens para llegar al 90% → los 200 se consideran.

La diferencia clave: top-k es estático (siempre k candidatos), top-p es adaptativo (el número de candidatos varía según la distribución de cada token).

**¿Qué valores de temperatura, y top-p cojo?**

Los valores por defecto de los parámetros de temperatura y top-p en la API de OpenAI son 1. Teniendo ambos valores por defecto a 1, el modelo muestrea tokens libremente según su distribución de probabilidad natural, sin restricciones adicionales.

Atendiendo a los distintos tipos de preguntas que queremos generar, sabemos que para los tipos factual_true y factual_reasoning, lo ideal es tener diversidad en la formulación de la pregunta, pero a su vez precisión factual en la que apoyarse para la respuesta, es decir el modelo debe generar preguntas variadas que no suenen repetitivas, pero las respuestas deben estar bien sustentadas en el documento. Para las preguntas con trampa (factual_trap) es necesario cierta creatividad controlada, para formular premisas falsas pero plausibles y sin volverse incoherente, para que la trampa sea verosímil, no extravagante.
Entonces para estos 3 tipos de preguntas factuales, lo conveniente es establecer una temperatura de 0.7 y un top-p por defecto de 1.

Para la generación de preguntas de fuera del dominio, necesitamos preguntas razonables que un empleado pudiera hacerse, pero que su respuesta no se encuentre en el documento. Por lo tanto necesitamos mas creatividad que con las pregutnas factuales, por lo que aplicamos un valor de temperatuta más alto, para que el modelo explore temas adyacentes al documento sin quedarse solo en lo que ya aparece. Por lo tanto podríamos fijar la temperatura para este caso en 0.85.



| Tipo de prompt    | temperature | top_p         | Razón                                          |
| ----------------- | ----------- | ------------- | ---------------------------------------------- |
| factual_true      | 0.7         | 1.0 (default) | Diversidad léxica sin perder precisión factual |
| factual_reasoning | 0.7         | 1.0 (default) | Igual: variedad pero coherencia con el texto   |
| factual_trap      | 0.7         | 1.0 (default) | Trampa verosímil, no extravagante              |
| out_of_domain     | 0.85         | 1.0 (default) | Exploración de temas adyacentes al documento   |


**En todos los casos se deja al parámetro top-p por defecto (1.0)

**Prompt 1: factual_true**

In [ ]:


"""
You are an expert dataset designer for RAG (Retrieval-Augmented Generation) evaluation systems.

USE CONTEXT:
The document below belongs to the internal knowledge base of AMC Global, a company specialized
in natural juices and beverages. Employees from any department or role (production, logistics,
HR, quality, sustainability, legal, administration, etc.) consult this knowledge base to resolve
day-to-day work questions. The questions you generate must reflect realistic queries that any
of these employees might ask in a natural work situation.

Your task is to generate exactly {n_preguntas} factual questions based on the document provided below.

DEFINITION:
A factual question must meet ALL of the following criteria:
- It has a clear, verifiable answer that can be found directly in the document.
- It asks about a specific fact, datum, procedure, rule, date, quantity, or named entity present in the text.
- It does NOT require inference or reasoning beyond reading the relevant fragment.

DIVERSITY REQUIREMENTS — MANDATORY:
- Each question must be based on a DIFFERENT section or topic of the document.
  Do NOT generate multiple questions about the same paragraph or subject.
- Each question must use a DIFFERENT grammatical structure and formulation style.
  Vary between: direct questions (¿Qué...?), indirect questions (¿Cuál es...?),
  procedural questions (¿Cómo se...?), conditional questions (¿En qué caso...?),
  and quantitative questions (¿Cuántos/as...?).
- Do NOT repeat the same sentence pattern across different questions.

LANGUAGE RULES — STRICTLY ENFORCED:
- NEVER use expressions like "según el documento", "según el manual", "el documento establece",
  "de acuerdo con el texto", "el documento indica", or any similar meta-documentary reference.
  Questions and answers must sound like natural communication between employees and a knowledge system,
  not like academic exercises about a text.
- Questions must be written as if the employee does NOT know the answer and is asking a work colleague
  or a company knowledge assistant. Use first or third person naturally.
  BAD:  "¿Qué tipo de ingredientes deben evitarse según el documento?"
  GOOD: "¿Qué ingredientes no pueden llevar los productos de AMC?"
  BAD:  "¿Según el manual, cuántos días de vacaciones tiene un trabajador?"
  GOOD: "¿Cuántos días de vacaciones me corresponden como empleado de AMC?"

OUTPUT FORMAT:
Return ONLY a valid JSON object. Do not include any explanation, commentary, or text outside the JSON.
The JSON object must have exactly ONE key: "questions", whose value is an array of objects.
Each element must have exactly these three fields:
- "question": the question in the same language as the document, following the LANGUAGE RULES above.
- "answer_reference": a complete and informative answer that fully addresses the question.
    Do NOT summarize or paraphrase superficially — include ALL relevant details present in the document
    that are needed to give a thorough response. Write as if you are an internal knowledge assistant
    providing a useful, actionable answer to an AMC Global employee.
- "context_gold": the EXACT and MINIMAL literal fragment from the document that supports the answer.
This must be a verbatim extract, not a paraphrase. Keep it as short as possible while fully supporting the answer.

{{
"questions": [
    {{
      "question": "...",
      "answer_reference": "...",
      "context_gold": "..."
    }}
]
}}

DOCUMENT:

{texto_completo}
"""


**Prompt 2: factual_reasoning**

In [ ]:
"""
You are an expert dataset designer for RAG (Retrieval-Augmented Generation) evaluation systems.

USE CONTEXT:
The document below belongs to the internal knowledge base of AMC Global, a company specialized
in natural juices and beverages. Employees from any department or role (production, logistics,
HR, quality, sustainability, legal, administration, etc.) consult this knowledge base to resolve
day-to-day work questions. The questions you generate must reflect realistic queries that require
connecting or reasoning about information from this document, as any of these employees might do.

Your task is to generate exactly {n_preguntas} reasoning-based questions from the document provided below.

DEFINITION:
A reasoning question must meet ALL of the following criteria:
- Its answer CANNOT be found in a single literal sentence of the document.
- It requires the reader to: compare two or more elements, infer a cause or consequence,
  apply a rule to a specific case, synthesize information from different parts of the text,
  or reason about an implicit relationship described in the document.
- The answer must still be grounded exclusively in the document content.
  Do NOT generate questions whose answers require external knowledge.

VALID REASONING SUBTYPES (use a variety of them):
- Comparison: "¿En qué se diferencia el procedimiento X del procedimiento Y en AMC?"
- Causal: "¿Por qué AMC exige que...?"
- Conditional: "Si como empleado me encuentro en la situación X, ¿qué debo hacer?"
- Multi-fragment synthesis: questions whose answer requires combining two or more separate sections.
- Consequence: "¿Qué implicaciones tendría para un trabajador de AMC que...?"

DIVERSITY REQUIREMENTS — MANDATORY:
- Each question must address a DIFFERENT section or topic of the document.
  Do NOT generate multiple questions about the same paragraph or subject.
- Use a DIFFERENT reasoning subtype for each question when possible.
- Do NOT repeat the same sentence pattern across different questions.

LANGUAGE RULES — STRICTLY ENFORCED:
- NEVER use expressions like "según el documento", "según el manual", "el documento establece",
  "de acuerdo con el texto", "el documento indica", or any similar meta-documentary reference.
  Questions and answers must sound like natural communication between employees and a knowledge system.
- Questions must be written as if the employee is genuinely trying to understand a work situation,
  not as if they are analyzing a text.
  BAD:  "¿Por qué razón el documento establece que los proveedores deben certificarse?"
  GOOD: "¿Por qué AMC exige que sus proveedores estén certificados?"
  BAD:  "¿Qué consecuencias tendría, según el documento, que un trabajador incumpla X?"
  GOOD: "¿Qué consecuencias puede tener para un empleado incumplir la norma X?"

OUTPUT FORMAT:
Return ONLY a valid JSON object. Do not include any explanation, commentary, or text outside the JSON.
The JSON object must have exactly ONE key: "questions", whose value is an array of objects.
Each element must have exactly these three fields:
- "question": the question in the same language as the document, following the LANGUAGE RULES above.
- "answer_reference": a complete and informative answer (3-5 sentences) that explicitly shows
  the reasoning chain used to reach the conclusion. Include ALL relevant details from the document.
  Do NOT give a superficial answer — explain the full logic as if answering an employee who needs
  to understand not just the answer but the reasoning behind it.
- "context_gold": the EXACT and MINIMAL literal fragment or fragments from the document
  that are necessary to construct the answer. If multiple fragments are needed,
  separate them with " [...] ".

{{
"questions": [
    {{
      "question": "...",
      "answer_reference": "...",
      "context_gold": "..."
    }}
]
}}

DOCUMENT:

{texto_completo}


"""

**Prompt 3: factual_trap**

In [ ]:
"""
You are an expert dataset designer for RAG (Retrieval-Augmented Generation) evaluation systems.

USE CONTEXT:
The document below belongs to the internal knowledge base of AMC Global, a company specialized
in natural juices and beverages. Employees from any department or role consult this knowledge base
daily. The trap questions you generate must simulate realistic misconceptions or wrong assumptions
that a real AMC Global employee might genuinely hold about their work context.

Your task is to generate exactly {n_preguntas} trap questions based on the document provided below.

DEFINITION:
A trap question must meet ALL of the following criteria:
- It contains a FALSE or MISLEADING premise embedded in the question itself.
- The false premise must directly contradict or distort information that IS present in the document.
- The question must sound plausible and natural — a real user could genuinely ask it believing the premise is true.
- The correct answer must CORRECT the false premise and provide the accurate information.
- The answer must be grounded exclusively in the document. Do NOT invent corrections.

TYPES OF TRAPS TO USE (vary between them):
- False numerical value: the question states a wrong quantity, date, or threshold.
  Example: "¿Todos los trabajadores tienen 30 días de vacaciones?" when the document says 22.
- False attribution: the question assigns a responsibility or action to the wrong person or department.
- False negation: the question implies something is NOT required when the document says it IS, or vice versa.
- False condition: the question implies a rule applies in a case where the document explicitly excludes it.
- Invented element: the question references a procedure, role, or term that does not exist in the document.

DIVERSITY REQUIREMENTS — MANDATORY:
- Each question must target a DIFFERENT section or topic of the document.
  Do NOT generate multiple trap questions about the same paragraph.
- Use a DIFFERENT trap type for each question when possible.
- Each trap must feel natural and believable — avoid traps that are obviously wrong.

LANGUAGE RULES — STRICTLY ENFORCED:
- NEVER use expressions like "según el documento", "según el manual", "el documento establece",
  "es cierto que" or any similar meta-documentary reference in questions or answers.
- Questions must sound like a genuine employee assumption or misunderstanding, not like
  an academic question about a text.
  BAD:  "¿Es correcto que, según el documento, los empleados tienen 30 días de vacaciones?"
  GOOD: "Tengo entendido que en AMC nos corresponden 30 días de vacaciones al año, ¿es correcto?"
  BAD:  "¿El documento indica que el responsable de X es el departamento Y?"
  GOOD: "¿No es el departamento Y el encargado de gestionar X en AMC?"


OUTPUT FORMAT:
Return ONLY a valid JSON object. Do not include any explanation, commentary, or text outside the JSON.
The JSON object must have exactly ONE key: "questions", whose value is an array of objects.
Each element must have exactly these three fields:
- "question": the trap question in the same language as the document, following the LANGUAGE RULES above.
- "answer_reference": an answer that explicitly corrects the false premise and provides the full
    accurate information available in the document (3-4 sentences). Do NOT just say "that is incorrect" — explain what
    the correct information is, including all relevant nuances present in the document.
- "context_gold": the EXACT and MINIMAL literal fragment from the document that allows the false premise
    to be identified and corrected. Must be verbatim. Keep it as short as possible.

{{
"questions": [
    {{
      "question": "...",
      "answer_reference": "...",
      "context_gold": "..."
    }}
]
}}

DOCUMENT:

{texto_completo}
"""


**Prompt 4: out_of_domain**

In [ ]:

"""

You are an expert dataset designer for RAG (Retrieval-Augmented Generation) evaluation systems.

USE CONTEXT:
The document below belongs to the internal knowledge base of AMC Global, a company specialized
in natural juices and beverages. Employees from any department consult this knowledge base daily.
The out-of-domain questions you generate must simulate realistic questions that an AMC Global
employee might ask but whose answers are simply not covered in this particular document.

Your task is to generate exactly {n_preguntas} out-of-domain questions based on the document provided below.

DEFINITION:
An out-of-domain question must meet ALL of the following criteria:
- Its answer is NOT present anywhere in the document, not even partially or implicitly.
- It is a PLAUSIBLE and NATURAL question that a real AMC Global employee might ask.
It must be thematically related to the general domain of the document (same organization, same topic area),
but address a specific aspect that the document simply does not cover.
- It must NOT be answerable by inference or reasoning from the document content.
- It must NOT be an obviously absurd or unrelated question.

TYPES OF OUT-OF-DOMAIN QUESTIONS TO USE (vary between them):
- Information about related processes not mentioned in the document.
- Specific data (figures, dates, names) of the organization not included in the text.
- Regulations or rules from external bodies referenced contextually but not explained.
- Operational details that would naturally complement the document but are absent from it.
- Follow-up questions a user would ask after reading the document, whose answers are not in it.

DIVERSITY REQUIREMENTS — MANDATORY:
- Each question must target a DIFFERENT thematic area relative to the document.
- Use a DIFFERENT question type from the list above for each question when possible.
- Do NOT repeat the same sentence structure across different questions.

ANGUAGE RULES — STRICTLY ENFORCED:
- NEVER use expressions like "según el documento", "el documento no menciona", or any
  meta-documentary reference.
- Questions must sound like a genuine employee asking a natural work question, completely unaware
  of whether the answer is in any document or not.
  BAD:  "¿Qué información no cubre el documento sobre el proceso de selección?"
  GOOD: "¿Cuánto dura el proceso de selección habitual en AMC Global?"

OUTPUT FORMAT:
Return ONLY a valid JSON object. Do not include any explanation, commentary, or text outside the JSON.
The JSON object must have exactly ONE key: "questions", whose value is an array of objects.
Each element must have exactly these TWO fields and no others:
- "question": the out-of-domain question in the same language as the document,
    following the LANGUAGE RULES above.
- "answer_reference": always and exactly this fixed string (do not alter it):
  "No se puede responder con la información disponible en la documentación de AMC Global."

{{
"questions": [
    {{
      "question": "...",
      "answer_reference": "No se puede responder con la información disponible en la documentación de AMC Global."
    }}
]
}}

DOCUMENT:

{texto_completo}


"""


In [ ]:
##Función encargada de hacer la llamada a la API, esta llamada debe ser asíncrona.
# Aunque la función queremos que sea asíncrona, se debe crear un solo cliente para contactar con la API para todo el
# procesamiento. El cliente de OpenAI gestiona internamente las conexiones HTTP, y crear un cliente supone una operación
# costosa ya que abre conexiones, carga configuraciones, etc...

##La función retorna la lista de diccionarios con la pregunta, respuesta y contexto para el tipo de pregunta indicado
# según el prompt y para el documento con el que se trabaje
async def obtener_preguntas(client: AsyncOpenAI, prompt:str, tipo_preguntas: str, nombre_doc: str,
                             documento:str, n_preguntas: int, temp : float, model : str = 'gpt-4.1')-> dict[dict]:

  "Llamada asícnrona a la API de OpenAI, para que resuelva la consulta al modelo"
  response = await client.chat.completions.create(
      model = model,
      messages = [{"role" : "user",
                  "content" : prompt.format(n_preguntas = n_preguntas,
                                          texto_completo = documento),}],
      temperature = temp,
      response_format={"type": "json_object"}
  )

  data_response = json.loads(response.choices[0].message.content)["questions"]


  #################Este bloque de código es para securizar que el modelo no ha variado la respuesta que se le
  #indica que debe devolver para preguntas de fuera del dominio ###########################
  if tipo_preguntas == "out_of_domain":
      RESPUESTA_OOD = "No se puede responder con la información disponible en la documentación de AMC Global."
      for d in data_response:
          d["answer_reference"] = RESPUESTA_OOD
  ###########################################################################################################

  final_response = add_extra_fields(data_response, tipo_preguntas, nombre_doc, documento )


  return final_response

In [ ]:
def add_extra_fields(data_response:list[dict], tipo_preguntas:str, nombre_doc:str, documento:str )-> dict[dict]:

    final_response = []
    #Añado este ID para poder facilitar la búsqueda durante la revisión manual
    id_dict = {"factual_true":"ft", "factual_reasoning":"fr", "factual_trap":"ftrp", "out_of_domain":"ood"}

    for i, d in enumerate(data_response, start = 1):
      d["id"] = f"{id_dict[tipo_preguntas]}_{i}"
      d["category"] = tipo_preguntas
      d['doc_name'] = nombre_doc
      if tipo_preguntas != 'out_of_domain':
        d['context_validation'] = validar_contexto( context_gold = d["context_gold"],
                                                    texto_completo = documento)
      else:
        d["context_validation"] = {"valid": True, "score": None, "status": "no_context"}


      final_response.append(d)

    return final_response



In [ ]:
def normalizar_con_mapa(texto: str) -> tuple[str, list[int]]:
    """
    Normaliza el texto y devuelve también un mapa de posiciones.
    pos_map[i] = índice en texto original que corresponde al carácter i del texto normalizado.
    """
    texto_nfd = unicodedata.normalize("NFD", texto)

    nfd_to_orig = []
    for orig_idx, orig_char in enumerate(texto):
        nfd_expansion = unicodedata.normalize("NFD", orig_char)
        for _ in nfd_expansion:
            nfd_to_orig.append(orig_idx)

    result     = []
    pos_map    = []
    prev_space = False

    for nfd_idx, char in enumerate(texto_nfd):
        orig_idx   = nfd_to_orig[nfd_idx]
        char_lower = char.lower()

        if unicodedata.category(char) == "Mn":
            continue

        if char_lower.isspace():
            if not prev_space:
                result.append(' ')
                pos_map.append(orig_idx)
                prev_space = True
        else:
            result.append(char_lower)
            pos_map.append(orig_idx)
            prev_space = False

    norm_text = ''.join(result)
    l = len(norm_text) - len(norm_text.lstrip())
    r = len(norm_text.rstrip())

    return norm_text[l:r], pos_map[l:r]


def normalizar(texto: str) -> str:
    """Normaliza el texto para comparación: minúsculas, sin acentos, espacios normalizados.
    Wrapper sin mapa para usos donde solo se necesita el texto normalizado."""
    norm, _ = normalizar_con_mapa(texto)
    return norm


In [ ]:

def validar_contexto(context_gold: str, texto_completo: str, umbral: float = 0.85) -> dict:
    """
    Comprueba si context_gold proviene literalmente del texto_completo del documento.
    """
    context_norm        = normalizar(context_gold)
    doc_norm, pos_map   = normalizar_con_mapa(texto_completo)   # CAMBIO: ahora con mapa

    len_context = len(context_norm)
    len_doc     = len(doc_norm)

    if len_context > len_doc:
        return {"valid": False, "score": 0.0, "status": "invalid", "best_match": ""}

    paso         = max(1, len_context // 4)
    mejor_score  = 0.0
    mejor_inicio = 0

    for inicio in range(0, len_doc - len_context + 1, paso):
        ventana = doc_norm[inicio: inicio + len_context]
        score   = difflib.SequenceMatcher(None, context_norm, ventana).ratio()
        if score > mejor_score:
            mejor_score  = score
            mejor_inicio = inicio          # CAMBIO: guardamos el índice, no el fragmento aún
        if mejor_score == 1.0:
            break

    # CAMBIO: recuperar el fragmento original usando el mapa de posiciones
    orig_start  = pos_map[mejor_inicio]
    ultimo_idx  = min(mejor_inicio + len_context - 1, len(pos_map) - 1)
    orig_end    = pos_map[ultimo_idx] + 1
    mejor_fragmento = texto_completo[orig_start:orig_end]

    if mejor_score >= umbral:
        status = "valid"
    elif mejor_score >= 0.70:
        status = "review"
    else:
        status = "invalid"

    return {
        "valid":      mejor_score >= umbral,
        "score":      round(mejor_score, 4),
        "status":     status,
        "best_match": mejor_fragmento          # ahora sí apunta al fragmento correcto
    }

# MAIN

In [ ]:
#Función para la extracción de la configuración desde el archivo .yml
def load_config(config_file): # Parameterize config file name
    """
    Carga la configuración desde el fichero YAML.
    """
    try:
        with open(config_file, "r") as f:
            config = yaml.safe_load(f)
        print("Configuración cargada exitosamente.")
        return config
    except Exception as e:
        print(f"Error al cargar la configuración: {e}")
        raise

In [ ]:
async def main():
  #Obtención del diccionario de configuración
  config = load_config("/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/GeneraciónBateríaPreguntas/cnf.yaml")
  #Parámetros de rutas y localización de ficheros
  input_cfg = config.get("input")
  input_path = input_cfg.get("path")
  file_extension = input_cfg.get("file_pattern")
  output_path = config.get("output").get("path")

  #Parámetros de configuración de la API y peticiones al modelo
  api = config.get('api')
  token_api = api.get('key')
  model = api.get("model")
  client = AsyncOpenAI(api_key=token_api)
  prompts = config.get("prompts")

  #Listado de documentos cuyo texto ya fue extraido y procesado, de los cuales extraemos las
  #preguntas-respuestas-contexto
  documents = sorted([
        f for f in os.listdir(input_path)
        if f.endswith(file_extension[0])
    ])

  all_questions = []
  output_dir_individual = os.path.join(output_path, "por_documento_single-turn")
  os.makedirs(output_dir_individual, exist_ok=True)



  for json_file in documents:

      #Comprobamos que el documento que vamos a procesar no tenga ya sus preguntas...
      nombre_salida = json_file.replace(".json", "_questions.json")
      ruta_salida = os.path.join(output_dir_individual, nombre_salida)

      if os.path.exists(ruta_salida):
        #Sacar esta lógica fuera para poder cargar las preguntas que ya tenemos de los ficheros en una variable a convertir a df.
        print(f"Ya procesado: {json_file}, cargando desde disco...")
        with open(ruta_salida, "r", encoding="utf-8") as f:
          doc_questions = json.load(f)["preguntas"]
          all_questions.extend(doc_questions)

        continue

      file_path = os.path.join(input_path, json_file)

      with open(file_path, "r", encoding="utf-8") as f:
          data = json.load(f)

      texto_completo = data["texto_completo"]
      nombre_original = data["nombre_original"]
      num_paginas = data["num_paginas"]

      print(nombre_original, "pag: ",num_paginas)
      diccionario_preguntas_tipo = calcular_num_preguntas(texto_completo)
      print(diccionario_preguntas_tipo)
      print("-"*30)

      (factual_true_questions,
       factual_reasoning_questions,
       factual_trap_questions,
       out_of_domain_questions)   = await asyncio.gather(
                                            #PETICIÓN API FACTUAL_TRUE
                                            obtener_preguntas(
                                            client = client,
                                            prompt=prompts.get("prompt_factual_true"),
                                            tipo_preguntas = "factual_true",
                                            nombre_doc = nombre_original,
                                            documento = texto_completo,
                                            n_preguntas = diccionario_preguntas_tipo.get("factual_true"),
                                            temp = prompts.get("temperature").get("factual_true"),
                                            model = model,
                                            ),
                                            #PETICIÓN API FACTUAL_REASONING
                                            obtener_preguntas(
                                            client = client,
                                            prompt=prompts.get("prompt_factual_reasoning"),
                                            tipo_preguntas = "factual_reasoning",
                                            nombre_doc = nombre_original,
                                            documento = texto_completo,
                                            n_preguntas = diccionario_preguntas_tipo.get("factual_reasoning"),
                                            temp = prompts.get("temperature").get("factual_reasoning"),
                                            model = model,
                                            ),
                                            #PETICIÓN API FACTUAL_TRAP
                                            obtener_preguntas(
                                            client = client,
                                            prompt=prompts.get("prompt_factual_trap"),
                                            tipo_preguntas = "factual_trap",
                                            nombre_doc = nombre_original,
                                            documento = texto_completo,
                                            n_preguntas = diccionario_preguntas_tipo.get("factual_trap"),
                                            temp = prompts.get("temperature").get("factual_trap"),
                                            model = model,
                                            ),
                                            #PETICIÓN API OUT_OF_DOMAIN
                                            obtener_preguntas(
                                            client = client,
                                            prompt=prompts.get("prompt_out_of_domain"),
                                            tipo_preguntas = "out_of_domain",
                                            nombre_doc = nombre_original,
                                            documento = texto_completo,
                                            n_preguntas = diccionario_preguntas_tipo.get("out_of_domain"),
                                            temp = prompts.get("temperature").get("out_of_domain"),
                                            model = model,
                                            )
                                        )
      #Agrupamos todas las listas devueltas en una misma.
      #Así la variable doc_questións será una lista de diccionarios con todas las preguntas, respuestas y contextos
      num_preguntas_factual_true = len(factual_true_questions)
      num_preguntas_factual_reasoning = len(factual_reasoning_questions)
      num_preguntas_factual_trap = len(factual_trap_questions)
      num_preguntas_ood = len(out_of_domain_questions)

      total_preguntas = num_preguntas_factual_true + num_preguntas_factual_reasoning + num_preguntas_factual_trap + num_preguntas_ood

      doc_questions = (factual_true_questions + factual_reasoning_questions + factual_trap_questions + out_of_domain_questions)

      json_a_guardar = {
          "documento": nombre_original,
          "total_preguntas": total_preguntas,
          "num_preguntas_factual_true": num_preguntas_factual_true,
          "num_preguntas_factual_reasoning": num_preguntas_factual_reasoning,
          "num_preguntas_factual_trap": num_preguntas_factual_trap,
          "num_preguntas_out_of_domain": num_preguntas_ood,
          "preguntas": doc_questions
      }




      with open(ruta_salida, "w", encoding="utf-8") as f:
          json.dump(json_a_guardar, f, ensure_ascii=False, indent=2)

      all_questions.extend(doc_questions)




  # return all_questions








In [ ]:
await main()

In [ ]:
def create_final_dataset(lista_preguntas: list[dict], output_path:str)-> list[dict]:
  df = pd.DataFrame(lista_preguntas)
  df = df.rename(columns={"id": "id_generacion"})
  df = df.sample(frac=1, random_state=42).reset_index(drop=True)
  # ID con formato Q001, Q002, ... para facilitar la lectura
  df.insert(0, "id", [f"Q{str(i+1).zfill(3)}" for i in range(len(df))])
  output_dir = os.path.join(output_path, "CSV_final")
  ruta_csv = os.path.join(output_dir, "dataset_preguntas_individuales.csv")
  df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import json

# Cargar el CSV
df = pd.read_csv("golden_dataset_final.csv", encoding="utf-8-sig")

# Opción 1: JSON como lista de diccionarios (el formato más natural para tu caso)
lista_preguntas = df.to_dict(orient="records")

# Opción 2: Guardar ese JSON a disco
with open("golden_dataset_final.json", "w", encoding="utf-8") as f:
    json.dump(lista_preguntas, f, ensure_ascii=False, indent=2)


## Acumulación en un único fichero.

In [ ]:


def split_context_gold(context_gold: str) -> list[str]:
    """Separa fragmentos unidos por [...] limpiando espacios residuales."""
    fragments = re.split(r'\[\.\.\.\]', context_gold)
    return [f.strip() for f in fragments if f.strip()]


def buid_a_single_jsonL()-> None:
  config = load_config("/content/drive/MyDrive/Colab Notebooks/Cuarto curso/TFG/GeneraciónBateríaPreguntas/cnf.yaml")
  output_path = config.get("output").get("path")
  output_dir_individual = os.path.join(output_path, "por_documento_single-turn")
  all_questions = []

  for filename in sorted(os.listdir(output_dir_individual)):

      ruta_ficheros = os.path.join(output_dir_individual, filename)
      if os.path.exists(ruta_ficheros):
        #Sacar esta lógica fuera para poder cargar las preguntas que ya tenemos de los ficheros en una variable a convertir a df.
        print(f"Extrayendo preguntas de : {filename}")
        with open(ruta_ficheros, "r", encoding="utf-8") as f:
          doc_questions = json.load(f)["preguntas"]
          for q in doc_questions:
            q.pop("context_validation", None)

          all_questions.extend(doc_questions) #Acaba siendo una lista de diccionarios con todas las preguntas. De esta lista nos tenemos que deshacer del campo context_validation y además sustituir el id por otro más consistente a nivel global

  for i, q in enumerate(all_questions):
      q["id"] = f"Q{str(i + 1).zfill(3)}"

  output_file = os.path.join(output_path, "golden_dataset_single-turn.jsonl")
  with open(output_file, "w", encoding="utf-8") as out_f:
      for q in all_questions:
          ragas_sample = {
              "user_input": q.get("question"),
              "reference_contexts":  split_context_gold(q.get("context_gold", "")), #q.get("context_gold").split("[...]"),  # RAGAS espera lista
              "reference": q.get("answer_reference"),
              "retrieved_contexts": None,  # Se rellena durante la evaluación RAG
              "response": None,            # Se rellena durante la evaluación RAG
              "metadata": {
                  "id": q.get("id"),
                  "category": q.get("category"),
                  "doc_name": q.get("doc_name"),
              }
          }
          out_f.write(json.dumps(ragas_sample, ensure_ascii=False) + "\n")

  print(f"Golden dataset guardado en: {output_file} ({len(all_questions)} muestras)")